# Custom Chatbot Project

TODO: In this cell, write an explanation of which dataset you have chosen and why it is appropriate for this task

I have chosen character description dataset for the following reasons:
1) It lets users ask question about characters in play, movie, reality show, musical etc
2) It can let user query about relationship between characters in a specific play, movie etc

## Data Wrangling

TODO: In the cells below, load your chosen dataset into a `pandas` dataframe with a column named `"text"`. This column should contain all of your text data, separated into at least 20 rows.

In [1]:
import openai
import pandas as pd
import numpy as np

In [2]:
openai.api_key = "YOUR API KEY"

In [3]:
!ls ./data/*

./data/2023_fashion_trends.csv	   ./data/nyc_food_scrap_drop_off_sites.csv
./data/character_descriptions.csv


In [4]:
!head ./data/character_descriptions.csv

In [5]:
df_char = pd.read_csv('./data/character_descriptions.csv')

In [6]:
df_char.Medium.value_counts()

Play              18
Reality Show       8
Musical            7
Movie              6
Sitcom             6
Limited Series     5
Opera              5
Name: Medium, dtype: int64

In [7]:
len(df_char), df_char.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55 entries, 0 to 54
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Name         55 non-null     object
 1   Description  55 non-null     object
 2   Medium       55 non-null     object
 3   Setting      55 non-null     object
dtypes: object(4)
memory usage: 1.8+ KB


(55, None)

In [8]:
df_char.iloc[2]

Name                                                       Alice
Description    A woman in her late 30s, Alice is a warm and n...
Medium                                                      Play
Setting                                                  England
Name: 2, dtype: object

In [9]:
df_char.Name.value_counts()

Emily                   1
Sir Toby Belch          1
Sophia                  1
Noah                    1
Chloe                   1
Jake                    1
Prince Lorenzo          1
Signora Rosa            1
Baron Gustavo           1
Francesca               1
Don Carlo               1
Duke Orsino             1
Lady Olivia             1
Malvolio                1
Maya                    1
Viola                   1
Sir Andrew Aguecheek    1
Bianca                  1
Sebastian               1
Feste                   1
Antonio                 1
Abigail                 1
Thomas                  1
Reverend Brown          1
Captain James           1
Mrs. Mercer             1
James                   1
Marcus                  1
Jack                    1
Will                    1
Alice                   1
Tom                     1
Sarah                   1
George                  1
Rachel                  1
John                    1
Maria                   1
Caleb                   1
Tyler       

In [10]:
df_char

,Name,Description,Medium,Setting
0,Emily,"A young woman in her early 20s, Emily is an as...",Play,England
1,Jack,"A middle-aged man in his 40s, Jack is a succes...",Play,England
2,Alice,"A woman in her late 30s, Alice is a warm and n...",Play,England
3,Tom,"A man in his 50s, Tom is a retired soldier and...",Play,England
4,Sarah,"A woman in her mid-20s, Sarah is a free-spirit...",Play,England
5,George,"A man in his early 30s, George is a charming a...",Play,England
6,Rachel,"A woman in her late 20s, Rachel is a shy and i...",Play,England
7,John,"A man in his 60s, John is a retired professor ...",Play,England
8,Maria,"A middle-aged Latina woman in her 40s, Maria i...",Movie,Texas
9,Caleb,"A young African American man in his early 20s,...",Movie,Texas


##### We will combine all the available information in single column called text consolidating all information with character description.
##### Column 'text' will used to create embeddings that will be used in formulation of RAG context

In [11]:
df_char['text'] = df_char.apply(lambda x: x['Name'] + " is a character in show type  " + x['Medium'] + " which is shot in location of " + x['Setting'] + ". " + x['Name'] + " is " + x['Description'], axis=1)

In [12]:
df_char['text'].iloc[0]

"Emily is a character in show type  Play which is shot in location of England. Emily is A young woman in her early 20s, Emily is an aspiring actress and Alice's daughter. She has a bubbly personality and a quick wit, but struggles with self-doubt and insecurity. She's also in a relationship with George."

In [13]:
df_char['text'].iloc[2]

"Alice is a character in show type  Play which is shot in location of England. Alice is A woman in her late 30s, Alice is a warm and nurturing mother of two, including Emily. She's kind-hearted and empathetic, but can be overly protective of her children and prone to worrying. She's married to Jack."

In [14]:
df_char['text'].iloc[1]

"Jack is a character in show type  Play which is shot in location of England. Jack is A middle-aged man in his 40s, Jack is a successful businessman and Sarah's boss. He has a no-nonsense attitude, but is fiercely loyal to his friends and family. He's married to Alice."

In [15]:
df_char[df_char['Name'] == "Sarah"]['text'].iloc[0]

"Sarah is a character in show type  Play which is shot in location of England. Sarah is A woman in her mid-20s, Sarah is a free-spirited artist and Jack's employee. She's creative, unconventional, and passionate about her work. However, she can also be flighty and impulsive at times."

In [16]:
df_char['text'].iloc[6]

"Rachel is a character in show type  Play which is shot in location of England. Rachel is A woman in her late 20s, Rachel is a shy and introverted librarian who is in a relationship with Tom. She's intelligent, thoughtful, and has a deep love of books. However, she struggles with social anxiety and often feels like an outsider."

In [17]:
df_char['text'].iloc[7]

"John is a character in show type  Play which is shot in location of England. John is A man in his 60s, John is a retired professor and Tom's father. He has a dry wit and a love of intellectual debate, but can also be stubborn and set in his ways."

In [18]:
##Setting embedding model, batch size
emb_model = "text-embedding-ada-002"
bs = 5
embeds= []
ds = len(df_char)
emb_model, bs, embeds, ds

('text-embedding-ada-002', 5, [], 55)

In [19]:
# let us create embeddings

for i in range(0, ds, bs):
    resp = openai.Embedding.create(
        input=df_char['text'].iloc[i:i+bs].tolist(),
        engine=emb_model
    )
    for data in resp['data']:
        data_emb = data['embedding']
        embeds.append(data_emb)

In [20]:
len(embeds)

55

In [21]:
len(embeds[0])

1536

In [22]:
df_char['embeds'] = embeds
df_char

,Name,Description,Medium,Setting,text,embeds
0,Emily,"A young woman in her early 20s, Emily is an as...",Play,England,Emily is a character in show type Play which ...,"[-0.014007964171469212, -0.008090022020041943,..."
1,Jack,"A middle-aged man in his 40s, Jack is a succes...",Play,England,Jack is a character in show type Play which i...,"[0.008103594183921814, -0.023805538192391396, ..."
2,Alice,"A woman in her late 30s, Alice is a warm and n...",Play,England,Alice is a character in show type Play which ...,"[0.011090533807873726, -0.0068944720551371574,..."
3,Tom,"A man in his 50s, Tom is a retired soldier and...",Play,England,Tom is a character in show type Play which is...,"[0.01612371765077114, -0.014847366139292717, 0..."
4,Sarah,"A woman in her mid-20s, Sarah is a free-spirit...",Play,England,Sarah is a character in show type Play which ...,"[-0.006663069594651461, -0.022750362753868103,..."
5,George,"A man in his early 30s, George is a charming a...",Play,England,George is a character in show type Play which...,"[-0.010464967228472233, -0.008763831108808517,..."
6,Rachel,"A woman in her late 20s, Rachel is a shy and i...",Play,England,Rachel is a character in show type Play which...,"[0.00021629472030326724, -0.011215937323868275..."
7,John,"A man in his 60s, John is a retired professor ...",Play,England,John is a character in show type Play which i...,"[0.01777494139969349, -0.01825176551938057, -0..."
8,Maria,"A middle-aged Latina woman in her 40s, Maria i...",Movie,Texas,Maria is a character in show type Movie which...,"[-0.007858765311539173, -0.01962009072303772, ..."
9,Caleb,"A young African American man in his early 20s,...",Movie,Texas,Caleb is a character in show type Movie which...,"[0.010566802695393562, -0.0329563245177269, 0...."


In [23]:
df_char.to_csv('./char_proj.csv')

## Custom Query Completion

TODO: In the cells below, compose a custom query using your chosen dataset and retrieve results from an OpenAI `Completion` model. You may copy and paste any useful code from the course materials.

In [24]:
## we will use the following 3 questions in this project:
##1)"Who is Emily?"
##2)"How are Alice, Jack and Emily related?"
##3)"What are the names of 8 characters in show type Play shot in England?"

In [25]:
q1 = "Who is Emily?"
q2 = "How are Alice, Jack and Emily related?"
q3 = "What are the names of 8 characters in show type Play shot in England?"
q1,q2,q3

('Who is Emily?',
 'How are Alice, Jack and Emily related?',
 'What are the names of 8 characters in show type Play shot in England?')

In [26]:
def create_q_embed(q):
    question = q
    q_emb = openai.Embedding.create(
        engine=emb_model,
        input=question
        )
    return q_emb['data'][0]['embedding']

In [27]:
q1_emb = create_q_embed(q1)
q2_emb = create_q_embed(q2)
q3_emb = create_q_embed(q3)

In [28]:
len(q1_emb), type(q1_emb), len(q2_emb), type(q2_emb), len(q3_emb), type(q3_emb)

(1536, list, 1536, list, 1536, list)

In [29]:
from openai.embeddings_utils import get_embedding, distances_from_embeddings

In [30]:
# Let us calculate cosine distance of question embeddings from different embeddings present in text column of dataframe
def calc_cos_dist(q_emb):
    cos_dist = distances_from_embeddings(q_emb,
                                         df_char['embeds'].values,
                                         distance_metric="cosine"
                                         )
    return cos_dist
#type(cos_dist), len(cos_dist)

In [31]:
q1_cos_dist = calc_cos_dist(q1_emb)
q2_cos_dist = calc_cos_dist(q2_emb)
q3_cos_dist = calc_cos_dist(q3_emb)

In [32]:
type(q1_cos_dist), len(q1_cos_dist), type(q2_cos_dist), len(q2_cos_dist), type(q3_cos_dist), len(q3_cos_dist)

(list, 55, list, 55, list, 55)

In [33]:
df_char['q1_cos_dist'] = q1_cos_dist
df_char['q2_cos_dist'] = q2_cos_dist
df_char['q3_cos_dist'] = q3_cos_dist

In [34]:
df_char

,Name,Description,Medium,Setting,text,embeds,q1_cos_dist,q2_cos_dist,q3_cos_dist
0,Emily,"A young woman in her early 20s, Emily is an as...",Play,England,Emily is a character in show type Play which ...,"[-0.014007964171469212, -0.008090022020041943,...",0.117426,0.177256,0.182400
1,Jack,"A middle-aged man in his 40s, Jack is a succes...",Play,England,Jack is a character in show type Play which i...,"[0.008103594183921814, -0.023805538192391396, ...",0.243633,0.189879,0.174346
2,Alice,"A woman in her late 30s, Alice is a warm and n...",Play,England,Alice is a character in show type Play which ...,"[0.011090533807873726, -0.0068944720551371574,...",0.180261,0.151564,0.183976
3,Tom,"A man in his 50s, Tom is a retired soldier and...",Play,England,Tom is a character in show type Play which is...,"[0.01612371765077114, -0.014847366139292717, 0...",0.237824,0.255482,0.198132
4,Sarah,"A woman in her mid-20s, Sarah is a free-spirit...",Play,England,Sarah is a character in show type Play which ...,"[-0.006663069594651461, -0.022750362753868103,...",0.223287,0.234086,0.176460
5,George,"A man in his early 30s, George is a charming a...",Play,England,George is a character in show type Play which...,"[-0.010464967228472233, -0.008763831108808517,...",0.204115,0.241372,0.185308
6,Rachel,"A woman in her late 20s, Rachel is a shy and i...",Play,England,Rachel is a character in show type Play which...,"[0.00021629472030326724, -0.011215937323868275...",0.217207,0.251718,0.199804
7,John,"A man in his 60s, John is a retired professor ...",Play,England,John is a character in show type Play which i...,"[0.01777494139969349, -0.01825176551938057, -0...",0.256561,0.251161,0.188895
8,Maria,"A middle-aged Latina woman in her 40s, Maria i...",Movie,Texas,Maria is a character in show type Movie which...,"[-0.007858765311539173, -0.01962009072303772, ...",0.248914,0.260889,0.236210
9,Caleb,"A young African American man in his early 20s,...",Movie,Texas,Caleb is a character in show type Movie which...,"[0.010566802695393562, -0.0329563245177269, 0....",0.277489,0.285446,0.253314


In [35]:
#df_char.sort_values(by="cd", ascending=True, inplace=True)
#df_char

In [36]:
df_char[df_char['Setting']=='England']

,Name,Description,Medium,Setting,text,embeds,q1_cos_dist,q2_cos_dist,q3_cos_dist
0,Emily,"A young woman in her early 20s, Emily is an as...",Play,England,Emily is a character in show type Play which ...,"[-0.014007964171469212, -0.008090022020041943,...",0.117426,0.177256,0.182400
1,Jack,"A middle-aged man in his 40s, Jack is a succes...",Play,England,Jack is a character in show type Play which i...,"[0.008103594183921814, -0.023805538192391396, ...",0.243633,0.189879,0.174346
2,Alice,"A woman in her late 30s, Alice is a warm and n...",Play,England,Alice is a character in show type Play which ...,"[0.011090533807873726, -0.0068944720551371574,...",0.180261,0.151564,0.183976
3,Tom,"A man in his 50s, Tom is a retired soldier and...",Play,England,Tom is a character in show type Play which is...,"[0.01612371765077114, -0.014847366139292717, 0...",0.237824,0.255482,0.198132
4,Sarah,"A woman in her mid-20s, Sarah is a free-spirit...",Play,England,Sarah is a character in show type Play which ...,"[-0.006663069594651461, -0.022750362753868103,...",0.223287,0.234086,0.176460
5,George,"A man in his early 30s, George is a charming a...",Play,England,George is a character in show type Play which...,"[-0.010464967228472233, -0.008763831108808517,...",0.204115,0.241372,0.185308
6,Rachel,"A woman in her late 20s, Rachel is a shy and i...",Play,England,Rachel is a character in show type Play which...,"[0.00021629472030326724, -0.011215937323868275...",0.217207,0.251718,0.199804
7,John,"A man in his 60s, John is a retired professor ...",Play,England,John is a character in show type Play which i...,"[0.01777494139969349, -0.01825176551938057, -0...",0.256561,0.251161,0.188895


In [37]:
len(df_char[df_char['Setting']=='England'])

8

In [38]:
df_char[df_char['Setting']=='England']['Name'].tolist()

['Emily', 'Jack', 'Alice', 'Tom', 'Sarah', 'George', 'Rachel', 'John']

In [39]:
basic_prompt_template = """
Question: {}
Answer:
"""

print(basic_prompt_template)


Question: {}
Answer:



In [40]:
prompt_template = """
Answer the question based on the context below, and if the question cannot be answered based on the context, say "I don't know"

Context:

{}

---

Question: {}
Answer:"""

print(prompt_template)


Answer the question based on the context below, and if the question cannot be answered based on the context, say "I don't know"

Context:

{}

---

Question: {}
Answer:


In [41]:
import tiktoken

In [42]:
tokenizer = tiktoken.get_encoding("cl100k_base")
tokenizer

<Encoding 'cl100k_base'>

In [43]:
def create_prompt(df, question, prompt_template, q_cos_dist):
    q_len = len(tokenizer.encode(question))
    p_len = len(tokenizer.encode(prompt_template))
    ans_len = 400
    total_len = 4000
    thr_len = total_len - (q_len+p_len+ans_len)
    df_sort_q = df.sort_values(by=q_cos_dist, ascending=True)
    text_values = df_sort_q['text'].values
    cont_len = 0
    embeds = []
    for text in text_values:
        text_len = len(tokenizer.encode(text))
        cont_len += text_len
        embeds.append(text)
        if cont_len >= thr_len:
            break
    embeds_pp = "\n\n##\n\n".join(embeds)
    prompt = prompt_template.format(embeds_pp, question)
    return prompt

In [44]:
prompt_q1 = create_prompt(df_char, q1, prompt_template, "q1_cos_dist")
prompt_q2 = create_prompt(df_char, q2, prompt_template, "q2_cos_dist")
prompt_q3 = create_prompt(df_char, q3, prompt_template, "q3_cos_dist")

In [45]:
print(prompt_q1)


Answer the question based on the context below, and if the question cannot be answered based on the context, say "I don't know"

Context:

Emily is a character in show type  Play which is shot in location of England. Emily is A young woman in her early 20s, Emily is an aspiring actress and Alice's daughter. She has a bubbly personality and a quick wit, but struggles with self-doubt and insecurity. She's also in a relationship with George.

##

Alice is a character in show type  Play which is shot in location of England. Alice is A woman in her late 30s, Alice is a warm and nurturing mother of two, including Emily. She's kind-hearted and empathetic, but can be overly protective of her children and prone to worrying. She's married to Jack.

##

George is a character in show type  Play which is shot in location of England. George is A man in his early 30s, George is a charming and charismatic businessman who is in a relationship with Emily. He's ambitious, confident, and always looking f

In [46]:
print(prompt_q2)


Answer the question based on the context below, and if the question cannot be answered based on the context, say "I don't know"

Context:

Alice is a character in show type  Play which is shot in location of England. Alice is A woman in her late 30s, Alice is a warm and nurturing mother of two, including Emily. She's kind-hearted and empathetic, but can be overly protective of her children and prone to worrying. She's married to Jack.

##

Emily is a character in show type  Play which is shot in location of England. Emily is A young woman in her early 20s, Emily is an aspiring actress and Alice's daughter. She has a bubbly personality and a quick wit, but struggles with self-doubt and insecurity. She's also in a relationship with George.

##

Jack is a character in show type  Play which is shot in location of England. Jack is A middle-aged man in his 40s, Jack is a successful businessman and Sarah's boss. He has a no-nonsense attitude, but is fiercely loyal to his friends and family. 

In [47]:
print(prompt_q3)


Answer the question based on the context below, and if the question cannot be answered based on the context, say "I don't know"

Context:

Jack is a character in show type  Play which is shot in location of England. Jack is A middle-aged man in his 40s, Jack is a successful businessman and Sarah's boss. He has a no-nonsense attitude, but is fiercely loyal to his friends and family. He's married to Alice.

##

Sarah is a character in show type  Play which is shot in location of England. Sarah is A woman in her mid-20s, Sarah is a free-spirited artist and Jack's employee. She's creative, unconventional, and passionate about her work. However, she can also be flighty and impulsive at times.

##

Emily is a character in show type  Play which is shot in location of England. Emily is A young woman in her early 20s, Emily is an aspiring actress and Alice's daughter. She has a bubbly personality and a quick wit, but struggles with self-doubt and insecurity. She's also in a relationship with G

In [48]:
print(basic_prompt_template.format(q1))
print(basic_prompt_template.format(q2))
print(basic_prompt_template.format(q3))


Question: Who is Emily?
Answer:


Question: How are Alice, Jack and Emily related?
Answer:


Question: What are the names of 8 characters in show type Play shot in England?
Answer:



In [49]:
def create_response(prompt):
    model = "gpt-3.5-turbo-instruct"
    response = openai.Completion.create(
        model = model,
        prompt = prompt,
        max_tokens = 400
        )
    return response['choices'][0]['text'].strip()

In [50]:
def gen_basic_rag_response(q, rag_prompt):
    ## Basic prompt response
    basic_prompt = basic_prompt_template.format(q)
    response = create_response(basic_prompt)
    print(f'#### Basic Response is \n {response}')
    rag_response =  create_response(rag_prompt)
    print(f'#### RAG Response is \n {rag_response}')

## Custom Performance Demonstration

TODO: In the cells below, demonstrate the performance of your custom query using at least 2 questions. For each question, show the answer from a basic `Completion` model query as well as the answer from your custom query.

### Question 1

In [51]:
q1

'Who is Emily?'

In [52]:
gen_basic_rag_response(q1, prompt_q1)

#### Basic Response is 
 Emily is not a specific person. It can be any person of that name.
#### RAG Response is 
 A young woman in her early 20s, aspiring actress, and Alice's daughter who has a bubbly personality and a quick wit, but struggles with self-doubt and insecurity. She's also in a relationship with George.


### Question 2

In [53]:
q2

'How are Alice, Jack and Emily related?'

In [54]:
gen_basic_rag_response(q2, prompt_q2)

#### Basic Response is 
 There is not enough information provided to determine their exact relationship. They could be siblings, friends, cousins, or even strangers.
#### RAG Response is 
 Alice is Emily's mother, Jack is Emily's father, and they are married.


### Question 3

In [55]:
q3

'What are the names of 8 characters in show type Play shot in England?'

In [56]:
gen_basic_rag_response(q3, prompt_q3)

#### Basic Response is 
 1. Romeo 
2. Juliet 
3. Hamlet 
4. Ophelia 
5. Macbeth 
6. Lady Macbeth 
7. King Lear 
8. Miranda
#### RAG Response is 
 Jack, Sarah, Emily, Alice, George, John, Tom, Rachel
